# Mancino Market: Predicting Daily Unit Sales
## Complete ensemble exercise | Business Analytics and Data Visualization
Prof. Jose Mendoza · Fall 2026

Can combining predictions improve daily sales forecasts? Fit four familiar regression models, combine their predictions using simple and weighted averages, and compare the results with individual models and a mean-sales baseline.

Mancino Market is a synthetic grocery retailer with five fictional Manhattan stores. One row represents one date, store, and product category. The target is numeric unit sales. Predictions can inform replenishment planning, but observed sales do not measure unmet demand during stockouts.

### How to use this notebook
1. Upload this `.ipynb` file at [Google Colab](https://colab.research.google.com/) using **File > Upload notebook**. A standard CPU runtime is sufficient.
2. Run sections 1–6 to prepare the data, baseline, and individual models.
3. Run section 7 to fit two averaging ensembles.
4. Compare validation results in section 8 and examine model errors in section 9.
5. After selection is complete, change `RUN_FINAL_TEST` to `True` in section 10 and evaluate once.

The notebook runs independently. No earlier notebooks or saved models are required. Data, predictors, and chronological splits match the companion exercises.


## 1. Import the tools
Colab includes these libraries. The notebook prints their versions so results can be traced to the environment used.


In [ ]:
import io
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False,
                     'axes.spines.right': False})
print('pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)


## 2. Load and inspect the data
Source: [mancino_daily_sales.csv](https://github.com/jrmst102/mancino/blob/main/data/v1_2025-11-01/mancino_daily_sales.csv).
The verified file contains 22,875 rows, covering January 1–November 1, 2025. The code reports the current file dimensions in case the repository changes.

| Field | Meaning | Use in this notebook |
|---|---|---|
| `date` | Selling date | Split data and derive calendar features |
| `store_id` | Store identifier | Categorical predictor |
| `category` | Product category | Categorical predictor |
| `day_of_week` | Weekday | Recomputed from the date |
| `avg_price` | Observed daily average price | Excluded |
| `units_sold` | Daily units for a store and category | Target |
| `revenue` | Realized sales revenue | Excluded |

Revenue contains information about the sales outcome and would leak information into the model. The observed average price is also excluded because its availability before the selling day has not been established. A verified, planned price could be added in a later exercise.


In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/jrmst102/mancino/main/data/v1_2025-11-01/mancino_daily_sales.csv'
LOCAL_CSV = Path('mancino_daily_sales.csv')

if LOCAL_CSV.exists():
    df = pd.read_csv(LOCAL_CSV)
    print('Loaded local CSV:', LOCAL_CSV)
else:
    try:
        df = pd.read_csv(DATA_URL)
        print('Loaded CSV from GitHub.')
    except Exception as error:
        print('GitHub download failed. Upload mancino_daily_sales.csv when prompted.')
        from google.colab import files
        uploaded = files.upload()
        if 'mancino_daily_sales.csv' not in uploaded:
            raise ValueError('Please upload the named Mancino CSV.') from error
        df = pd.read_csv(io.BytesIO(uploaded['mancino_daily_sales.csv']))

required = {'date', 'store_id', 'category', 'units_sold'}
assert required.issubset(df.columns), 'The CSV is missing required columns.'
df['date'] = pd.to_datetime(df['date'], errors='raise')
assert not df[list(required)].isna().any().any(), 'Check missing required values.'
assert not df.duplicated(['date', 'store_id', 'category']).any(), 'Check duplicate rows.'
assert pd.api.types.is_numeric_dtype(df['units_sold'])
assert df['units_sold'].ge(0).all(), 'Check negative unit sales.'
df = df.sort_values(['date', 'store_id', 'category']).reset_index(drop=True)
print(f'{len(df):,} rows | {df.store_id.nunique()} stores | {df.category.nunique()} categories')
print('Dates:', df.date.min().date(), 'to', df.date.max().date())
display(df.head())
display(df.isna().sum().rename('Missing values').to_frame())


## 3. Define predictors and split by date
Use only information available before the selling day: store, category, weekday, and day of year. Store numbers are identifiers, so they are treated as categories. Day of year supplies a simple numeric time feature; it is not a full seasonality model.

The earliest 60% of dates form the training set, the next 20% form the validation set, and the latest 20% form the test set. Every observation from a given date stays in the same set. This asks whether patterns learned from earlier dates carry forward to later dates for the existing stores and categories.

Train on the training set, choose settings using validation results, and open the test set only after those choices are complete. This notebook does not use lagged sales, promotions, or inventory data.


In [ ]:
df['store_id'] = df['store_id'].astype(str)
df['weekday'] = df['date'].dt.day_name()
df['day_of_year'] = df['date'].dt.dayofyear
CATEGORICAL = ['store_id', 'category', 'weekday']
NUMERIC = ['day_of_year']
FEATURES = CATEGORICAL + NUMERIC
TARGET = 'units_sold'

dates = np.sort(df['date'].unique())
train_end = dates[int(len(dates) * 0.60) - 1]
validation_end = dates[int(len(dates) * 0.80) - 1]
train = df.loc[df.date <= train_end].copy()
validation = df.loc[(df.date > train_end) & (df.date <= validation_end)].copy()
test = df.loc[df.date > validation_end].copy()
assert train.date.max() < validation.date.min() <= validation.date.max() < test.date.min()

X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = validation[FEATURES], validation[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
display(pd.DataFrame([
    {'Set': name, 'Rows': len(part), 'First date': part.date.min().date(),
     'Last date': part.date.max().date()}
    for name, part in [('Training', train), ('Validation', validation), ('Test', test)]
]))


In [ ]:
# Explore training outcomes only; keep later outcomes for evaluation.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(y_train, bins=30, color='#356DA8', edgecolor='white')
axes[0].set(xlabel='Daily units per store/category', ylabel='Number of observations',
            title='Training sales distribution')
train.groupby('date').units_sold.sum().plot(ax=axes[1], color='#356DA8')
axes[1].set(xlabel='Date', ylabel='Total units across stores/categories',
            title='Training sales over time')
plt.tight_layout()
plt.show()


## 4. Prepare a shared evaluation workflow
Each individual model has its own preprocessing pipeline. Categories are one-hot encoded. KNN and the neural network also standardize day of year; tree-based models use its original scale. All transformations are fitted within the training period only.

The neural network standardizes the target internally and returns predictions in original sales units. Averaging therefore combines predictions expressed in the same units. The baseline is a benchmark and is not a member of either ensemble.

| Metric | Interpretation | Preferred direction |
|---|---|---|
| MAE | Average absolute error, in units per store/category/day | Lower |
| RMSE | Error in units, with larger misses receiving more weight | Lower |
| R² | Performance relative to predicting the evaluation set's mean | Higher |

R² can be negative and is not a percentage of correct predictions. Use chronological validation MAE to select the preferred approach. Ensemble fitting time includes refitting all its members; prediction time is not included.


In [ ]:
def make_pipeline(estimator, scale_numeric=False):
    preparation = ColumnTransformer([
        ('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL),
        ('numeric', StandardScaler() if scale_numeric else 'passthrough', NUMERIC)
    ])
    return Pipeline([('prepare', preparation), ('model', estimator)])

def metrics(actual, predicted):
    return {'MAE': mean_absolute_error(actual, predicted),
            'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
            'R2': r2_score(actual, predicted)}

# Rerunning this cell clears previous results.
models, results, train_predictions, validation_predictions = {}, {}, {}, {}

def fit_and_evaluate(name, estimator):
    fitted = clone(estimator)
    started = time.perf_counter()
    fitted.fit(X_train, y_train)
    seconds = time.perf_counter() - started
    train_predictions[name] = fitted.predict(X_train)
    validation_predictions[name] = fitted.predict(X_val)
    train_scores = metrics(y_train, train_predictions[name])
    val_scores = metrics(y_val, validation_predictions[name])
    models[name] = fitted
    results[name] = {'Model': name, 'Train MAE': train_scores['MAE'],
                     'Validation MAE': val_scores['MAE'],
                     'Validation RMSE': val_scores['RMSE'],
                     'Validation R2': val_scores['R2'], 'Fit seconds': seconds}
    display(pd.DataFrame([results[name]]).round(3))
    return fitted


## 5. Establish a baseline
The baseline always predicts the training set's mean sales. A more complicated model must earn its place by improving on this simple rule. A later extension could use historical averages for each store, category, and weekday as a stronger baseline.


In [ ]:
baseline = fit_and_evaluate('Mean baseline', DummyRegressor(strategy='mean'))


## 6. Fit the individual models
Use one fixed configuration for each model so the exercise focuses on combining predictions rather than repeating the earlier tuning exercises.

| Model | Configuration |
|---|---|
| KNN | 50 neighbors, uniform weights |
| Regression tree | Maximum depth 12, minimum leaf size 30 |
| Random forest | 100 trees, maximum depth 12, minimum leaf size 30, `max_features=0.8` |
| Neural network | One hidden layer with 16 neurons, ReLU activation, Adam optimizer |

The network scales the target, uses `alpha=0.01`, and trains for at most 300 epochs. Early stopping monitors an internal random 15% holdout from the training period. That internal score is a training diagnostic; it does not replace the separate chronological validation set used here. Preprocessing has been fitted across the training period, including that internal holdout.

Keep these configurations fixed when comparing averaging rules. Random sampling uses a fixed seed. A random forest already averages trees; this exercise adds a second level of averaging across different model types.


In [ ]:
neural_estimator = TransformedTargetRegressor(
    regressor=MLPRegressor(
        hidden_layer_sizes=(16,), activation='relu', solver='adam',
        alpha=0.01, batch_size=256, learning_rate_init=0.001, max_iter=300,
        early_stopping=True, validation_fraction=0.15,
        n_iter_no_change=20, random_state=SEED),
    transformer=StandardScaler())

base_estimators = {
    'KNN': make_pipeline(KNeighborsRegressor(n_neighbors=50, weights='uniform', n_jobs=-1), True),
    'Regression tree': make_pipeline(DecisionTreeRegressor(
        max_depth=12, min_samples_leaf=30, random_state=SEED)),
    'Random forest': make_pipeline(RandomForestRegressor(
        n_estimators=100, max_depth=12, min_samples_leaf=30,
        max_features=0.8, bootstrap=True, random_state=SEED, n_jobs=-1)),
    'Neural network': make_pipeline(neural_estimator, True)
}
BASE_NAMES = list(base_estimators)

# Clear earlier ensemble results if the individual-model settings are rerun.
for name in list(models):
    if name.startswith('Ensemble:'):
        for collection in (models, results, train_predictions, validation_predictions):
            collection.pop(name, None)

for name, estimator in base_estimators.items():
    fit_and_evaluate(name, estimator)


## 7. Combine predictions
The equal-weight ensemble gives each model 25% of the prediction:

$$\hat{y}_{equal} = \frac{\hat{y}_{KNN}+\hat{y}_{tree}+\hat{y}_{forest}+\hat{y}_{network}}{4}.$$

For example, predictions of 20, 24, 28, and 32 units yield an ensemble prediction of 26 units.

The weighted ensemble uses relative weights `[1, 1, 2, 2]`: one-sixth each for KNN and the regression tree, and one-third each for the random forest and neural network. Those illustrative weights are specified before evaluation; they are not an optimized solution or a claim that either model deserves more weight.

`VotingRegressor` computes numeric averages for regression, despite its name. It fits fresh copies of each member pipeline on the same training data. All forecasts remain in units sold.

Averaging can help when errors partly offset. It may also dilute a stronger model with weaker predictions, so improvement is not guaranteed. Do not use test results to choose weights.


In [ ]:
WEIGHTS = [1, 1, 2, 2]  # Order: KNN, regression tree, random forest, neural network.
assert len(WEIGHTS) == len(BASE_NAMES)
assert np.isfinite(WEIGHTS).all() and np.all(np.asarray(WEIGHTS) >= 0) and sum(WEIGHTS) > 0

members = [(f'model_{i}', estimator) for i, estimator in enumerate(base_estimators.values())]
ensemble_estimators = {
    'Ensemble: equal weights': VotingRegressor(estimators=members, n_jobs=1),
    'Ensemble: weighted average': VotingRegressor(estimators=members, weights=WEIGHTS, n_jobs=1)
}
ENSEMBLE_NAMES = list(ensemble_estimators)
for name, estimator in ensemble_estimators.items():
    fit_and_evaluate(name, estimator)

display(pd.DataFrame({
    'Model': BASE_NAMES,
    'Equal share': np.repeat(1 / len(BASE_NAMES), len(BASE_NAMES)),
    'Weighted share': np.asarray(WEIGHTS) / sum(WEIGHTS)
}).round(3))

# Show exactly how the weighted ensemble combines one validation row.
weighted = models['Ensemble: weighted average']
one_row = X_val.iloc[[0]]
member_values = np.array([member.predict(one_row)[0] for member in weighted.estimators_])
manual_average = np.average(member_values, weights=WEIGHTS)
assert np.isclose(manual_average, weighted.predict(one_row)[0])
display(pd.DataFrame({'Member': BASE_NAMES, 'Predicted units': member_values}).round(2))
print('Weighted prediction for this row:', round(manual_average, 2), 'units')


## 8. Compare validation results and select an approach
The table compares the baseline, four individual models, and two ensembles on identical validation observations. Positive improvement percentages mean lower MAE than the baseline. Check whether the best ensemble also beats the best individual model.

The code records the best individual model, best ensemble, and overall preferred approach using validation MAE only. Exact ties retain the first listed candidate. If an ensemble improves MAE only slightly, consider whether maintaining four models is justified.

Which averaging rule performed better? Did either improve on the best individual model? If you change weights, keep the search small and rerun sections 7–9 before opening the test set. Repeated choices based on the same validation period can overfit that period.


In [ ]:
assert all(name in results for name in ['Mean baseline'] + BASE_NAMES + ENSEMBLE_NAMES)
comparison = pd.DataFrame(results.values()).sort_values(
    'Validation MAE', kind='stable').reset_index(drop=True)
baseline_mae = results['Mean baseline']['Validation MAE']
comparison['MAE improvement vs baseline (%)'] = (
    100 * (1 - comparison['Validation MAE'] / baseline_mae) if baseline_mae > 0 else np.nan)
display(comparison.round(3))

best_individual = min(BASE_NAMES, key=lambda name: results[name]['Validation MAE'])
best_ensemble = min(ENSEMBLE_NAMES, key=lambda name: results[name]['Validation MAE'])
preferred_name = min(results, key=lambda name: results[name]['Validation MAE'])
print('Best individual:', best_individual)
print('Best ensemble:', best_ensemble)
print('Preferred approach using validation MAE:', preferred_name)
print('Best individual MAE minus best ensemble MAE:',
      round(results[best_individual]['Validation MAE'] - results[best_ensemble]['Validation MAE'], 3))
print('A positive difference favors the ensemble; a negative difference favors the individual model.')

comparison.set_index('Model')[['Train MAE', 'Validation MAE']].plot.barh(
    color=['#A7C4DE', '#356DA8'], figsize=(10, 5))
plt.xlabel('MAE: units per store/category/day')
plt.title('Individual models and averaging ensembles')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 9. Examine whether model errors differ
Averaging has more opportunity to help when models make different errors. The matrix reports correlations between validation residuals, defined as actual minus predicted sales. Correlations close to 1 indicate that errors tend to move together. Lower correlations suggest more diversity, but neither low correlation nor a larger number of models guarantees better MAE.

All models can miss the same unpredictable sales variation, so correlated residuals are common. Use the error correlations alongside each model's MAE. Which pair makes the most similar errors? Does this help explain the ensemble's result? These are descriptive checks on validation data, not causal evidence or tests of statistical significance.


In [ ]:
residuals = pd.DataFrame({
    name: y_val.to_numpy() - validation_predictions[name] for name in BASE_NAMES
})
error_correlations = residuals.corr()
display(error_correlations.round(3))

fig, ax = plt.subplots(figsize=(7, 5))
chart = ax.imshow(error_correlations, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(BASE_NAMES)), BASE_NAMES, rotation=25, ha='right')
ax.set_yticks(range(len(BASE_NAMES)), BASE_NAMES)
for i in range(len(BASE_NAMES)):
    for j in range(len(BASE_NAMES)):
        value = error_correlations.iloc[i, j]
        ax.text(j, i, f'{value:.2f}', ha='center', va='center',
                color='white' if abs(value) > 0.7 else 'black')
ax.set_title('Correlations between validation errors')
fig.colorbar(chart, ax=ax, label='Residual correlation')
plt.tight_layout()
plt.show()


## 10. Final test: open only after selection
The switch below is off by default. After completing the validation comparisons, change it to `True`. Before opening test outcomes, the code fixes three approaches for reporting: the best ensemble, the best individual model, and the baseline. It refits them on training plus validation data, then evaluates them on the untouched test period.

The approach preferred in section 8 remains the validation-selected choice. The test comparison is a final assessment; do not switch weights, model settings, or selection rules based on these test results. Further tuning would require a new test period. Neural-network members retain their internal early-stopping holdout within the combined development period.

Predictions are evaluated without rounding or clipping. Complete this interpretation: “Our validation-selected approach was ___. Its test MAE was ___ units per store/category/day. The ensemble [did/did not] improve on the individual model, and maintaining four models [is/is not] justified because ___.”

Observed sales can be limited by stockouts. These synthetic patterns support learning the workflow, not claims about actual retail performance or causal effects of business decisions.


In [ ]:
RUN_FINAL_TEST = False  # Change to True only after completing sections 8 and 9.

if RUN_FINAL_TEST:
    assert all(name in globals() for name in ['best_individual', 'best_ensemble', 'preferred_name']),         'Run section 8 first.'
    report_names = list(dict.fromkeys([best_ensemble, best_individual, 'Mean baseline']))
    assert preferred_name in report_names
    X_development = pd.concat([X_train, X_val])
    y_development = pd.concat([y_train, y_val])
    final_models, final_predictions, test_rows = {}, {}, []
    for name in report_names:
        final_models[name] = clone(models[name]).fit(X_development, y_development)
        final_predictions[name] = final_models[name].predict(X_test)
        test_rows.append({'Model': name, **metrics(y_test, final_predictions[name])})
    print('Approach selected before opening test outcomes:', preferred_name)
    display(pd.DataFrame(test_rows).round(3))

    prediction_examples = test[['date', 'store_id', 'category', TARGET]].copy()
    for name in report_names:
        prediction_examples[name] = final_predictions[name]
    display(prediction_examples.head(10).round(2))

    chosen_predictions = final_predictions[preferred_name]
    print('Negative predictions from the selected approach:', int(np.sum(chosen_predictions < 0)))
    plt.scatter(y_test, chosen_predictions, alpha=0.15, s=12, color='#356DA8')
    low = min(y_test.min(), chosen_predictions.min())
    high = max(y_test.max(), chosen_predictions.max())
    plt.plot([low, high], [low, high], color='#B64B44', linestyle='--')
    plt.xlabel('Actual units')
    plt.ylabel('Predicted units')
    plt.title(f'Final test: {preferred_name}')
    plt.tight_layout()
    plt.show()
else:
    print('Test results remain unopened. Complete the validation comparisons first.')
